## the objective of this notebook is to read data from zip files stored in a volume into appropriate delta tables to do downstream analysis on the Seattle market

Ingest Steps:
- read data in from zip files in raw_data schema and create appropriate delta tables
- Use catalog cmegdemos_catalog and network_analytics_enablement schema (parameterize these values throughout the notebook)
- raw data zip files are in the raw_data volume in this schema
- Load FCC BDC H3 hexagon coverage data - filter to only Seattle area
- Importing Microsoft building footprints as polygon geometries using Databricks Spatial SQL and save to delta table 
- load in 310.csv.gz file from Open Cell ID for tower data in seatle market but filter to only t mobile
- make sure tables are ready for downstream analysis with H3 and ST functions

In [0]:
%pip install pyshp --quiet

In [0]:
CATALOG = "cmegdemos_catalog"
SCHEMA = "network_analytics_enablement"
VOLUME = "raw_data"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Seattle metro area bounding box for spatial filtering
SEATTLE_BBOX = {
    "lat_min": 47.40, "lat_max": 47.80,
    "lon_min": -122.50, "lon_max": -122.10
}

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Volume path: {VOLUME_PATH}")
print(f"Seattle bounding box: {SEATTLE_BBOX}")

### Ingest FCC BDC H3 Hexagon Coverage Data
Read 5G NR mobile broadband coverage data from the FCC BDC geopackage, filter to Seattle metro area using H3 cell center coordinates, and save as a Delta table.

In [0]:
import sqlite3, zipfile, tempfile, os, shutil
import pandas as pd

# --- Extract GeoPackage from zip ---
tmpdir = tempfile.mkdtemp()
zf_path = f"{VOLUME_PATH}/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip"
with zipfile.ZipFile(zf_path) as zf:
    zf.extractall(tmpdir)

gpkg_path = os.path.join(tmpdir, "bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.gpkg")

# --- Read tabular data from GeoPackage (skip binary geometry) ---
conn = sqlite3.connect(gpkg_path)
pdf_bdc = pd.read_sql_query("""
    SELECT fid, technology, mindown, minup, environmnt, h3_res9_id 
    FROM bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026
""", conn)
conn.close()
shutil.rmtree(tmpdir)

print(f"Total BDC records in WA: {len(pdf_bdc):,}")

# --- Create Spark DataFrame and filter to Seattle ---
df_bdc = spark.createDataFrame(pdf_bdc)
df_bdc.createOrReplaceTempView("bdc_raw")

table_name = f"{CATALOG}.{SCHEMA}.fcc_bdc_h3_seattle"
spark.sql(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    WITH bdc_centers AS (
        SELECT *,
            ST_Y(ST_GeomFromWKB(h3_centeraswkb(h3_res9_id))) AS center_lat,
            ST_X(ST_GeomFromWKB(h3_centeraswkb(h3_res9_id))) AS center_lon
        FROM bdc_raw
        WHERE h3_isvalid(h3_res9_id)
    )
    SELECT fid, technology, mindown, minup, environmnt, h3_res9_id
    FROM bdc_centers
    WHERE center_lat BETWEEN {SEATTLE_BBOX['lat_min']} AND {SEATTLE_BBOX['lat_max']}
      AND center_lon BETWEEN {SEATTLE_BBOX['lon_min']} AND {SEATTLE_BBOX['lon_max']}
""")

count = spark.table(table_name).count()
print(f"Saved {count:,} Seattle-area records to {table_name}")
display(spark.table(table_name).limit(5))

### Ingest Microsoft Building Footprints
Read building footprint polygons from the Washington state shapefile, filter to Seattle metro area, and store as a Delta table with native `GEOMETRY(4326)` columns for spatial SQL operations.

In [0]:
import shapefile
import zipfile, tempfile, os, shutil
import pandas as pd

# --- Extract shapefile from zip ---
tmpdir = tempfile.mkdtemp()
with zipfile.ZipFile(f"{VOLUME_PATH}/Washington.zip") as zf:
    zf.extractall(tmpdir)

shp_path = os.path.join(tmpdir, "bldg_footprints")

# --- Helper: convert shapefile polygon to WKT ---
def shape_to_wkt(shape):
    """Convert a pyshp Shape (polygon) to WKT string."""
    if shape.shapeType != 5:  # 5 = Polygon
        return None
    parts = list(shape.parts) + [len(shape.points)]
    rings = []
    for i in range(len(parts) - 1):
        ring_pts = shape.points[parts[i]:parts[i + 1]]
        coords = ", ".join(f"{p[0]} {p[1]}" for p in ring_pts)
        rings.append(f"({coords})")
    return f"POLYGON({', '.join(rings)})"

# --- Read shapefile with bounding box filter for Seattle ---
sf = shapefile.Reader(shp_path)
records = []
for sr in sf.iterShapeRecords():
    bbox = sr.shape.bbox  # (xmin, ymin, xmax, ymax) = (lon_min, lat_min, lon_max, lat_max)
    if (bbox[0] <= SEATTLE_BBOX['lon_max'] and bbox[2] >= SEATTLE_BBOX['lon_min'] and
        bbox[1] <= SEATTLE_BBOX['lat_max'] and bbox[3] >= SEATTLE_BBOX['lat_min']):
        wkt = shape_to_wkt(sr.shape)
        if wkt:
            records.append({
                'height': float(sr.record[0]) if sr.record[0] else None,
                'wkt': wkt
            })

shutil.rmtree(tmpdir)
print(f"Building footprints in Seattle area: {len(records):,}")

# --- Create Delta table with GEOMETRY column ---
pdf_bldg = pd.DataFrame(records)
spark.createDataFrame(pdf_bldg).createOrReplaceTempView("buildings_raw")

table_name = f"{CATALOG}.{SCHEMA}.building_footprints"
spark.sql(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    SELECT
        monotonically_increasing_id() AS building_id,
        CAST(height AS DOUBLE) AS height,
        ST_GeomFromText(wkt, 4326) AS geometry
    FROM buildings_raw
""")

count = spark.table(table_name).count()
print(f"Saved {count:,} building footprints to {table_name}")
display(spark.sql(f"""
    SELECT building_id, height, ST_AsText(geometry) AS geometry_wkt
    FROM {table_name} LIMIT 5
"""))

### Ingest OpenCellID Cell Tower Data
Read real crowdsourced cell tower records from `310.csv.gz` (MCC 310 = USA, OpenCellID format), filter to T-Mobile (MNC 260) towers in the Seattle metro area, and store as a Delta table with `GEOMETRY(4326)` point locations.

In [0]:
from pyspark.sql.types import DoubleType, LongType, IntegerType
from pyspark.sql.functions import col, from_unixtime

# --- Read OpenCellID CSV (no header, 14 columns) ---
raw = (
    spark.read.format("csv")
    .option("header", "false")
    .load(f"{VOLUME_PATH}/310.csv.gz")
)

column_names = [
    "radio", "mcc", "net", "area", "cell", "unit",
    "lon", "lat", "range_m", "samples", "changeable",
    "created", "updated", "avg_signal"
]
df_ocid = (
    raw.toDF(*column_names)
    .select(
        col("radio"),
        col("mcc").cast(IntegerType()),
        col("net").cast(IntegerType()),
        col("area").cast(IntegerType()),
        col("cell").cast(LongType()),
        col("unit").cast(IntegerType()),
        col("lon").cast(DoubleType()),
        col("lat").cast(DoubleType()),
        col("range_m").cast(IntegerType()),
        col("samples").cast(IntegerType()),
        from_unixtime(col("created").cast(LongType())).alias("created_ts"),
        from_unixtime(col("updated").cast(LongType())).alias("updated_ts"),
        col("avg_signal").cast(IntegerType()),
    )
)

total = df_ocid.count()
print(f"Total OpenCellID records (MCC 310 USA): {total:,}")

# --- Filter to Seattle T-Mobile towers only (MNC 260) and save ---
df_ocid.createOrReplaceTempView("ocid_raw")

table_name = f"{CATALOG}.{SCHEMA}.cell_towers"
spark.sql(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    SELECT
        monotonically_increasing_id() AS tower_id,
        'T-Mobile' AS carrier,
        radio AS tower_type,
        area AS lac_tac,
        cell AS cell_id,
        range_m AS coverage_radius_m,
        samples,
        lat AS latitude,
        lon AS longitude,
        created_ts,
        updated_ts,
        avg_signal,
        ST_Point(lon, lat, 4326) AS location
    FROM ocid_raw
    WHERE net = 260
      AND lat BETWEEN {SEATTLE_BBOX['lat_min']} AND {SEATTLE_BBOX['lat_max']}
      AND lon BETWEEN {SEATTLE_BBOX['lon_min']} AND {SEATTLE_BBOX['lon_max']}
""")

count = spark.table(table_name).count()
print(f"Saved {count:,} Seattle T-Mobile cell towers to {table_name}")
display(spark.sql(f"""
    SELECT tower_id, carrier, tower_type, cell_id, coverage_radius_m,
           samples, latitude, longitude, ST_AsText(location) AS location_wkt
    FROM {table_name} LIMIT 10
"""))

In [0]:
# --- Validate all ingested tables ---
tables = [
    f"{CATALOG}.{SCHEMA}.fcc_bdc_h3_seattle",
    f"{CATALOG}.{SCHEMA}.building_footprints",
    f"{CATALOG}.{SCHEMA}.cell_towers",
]

print("=== Ingestion Summary ===")
for t in tables:
    df = spark.table(t)
    count = df.count()
    cols = df.columns
    print(f"\u2713 {t}: {count:,} rows, {len(cols)} columns")
    print(f"  Columns: {', '.join(cols)}")